<a href="https://colab.research.google.com/github/umang0015/Ai-DataScience/blob/main/Generative%20AI/31AugSession_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install faiss-gpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 13.7 MB/s eta 0:00:00


In [3]:
!pip install -q langchain-huggingface langchain-text-splitters

In [15]:
# import libraries
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace , HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
import torch

In [16]:
generator = pipeline("text-generation", model="Qwen/Qwen2-1.5B-Instruct",
                     torch_dtype=torch.bfloat16, device_map="auto")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [17]:
LLM=HuggingFacePipeline(pipeline=generator)

In [18]:
# chunks the document using

In [19]:
import os

In [20]:
!ls

 github_ai_repos_2026.csv  'Restaurant reviews.csv'	'train - train.csv'
 my_model.h5		   'SPAM text - SPAM text.csv'


In [21]:
# change working directory
os.chdir('/content/drive/MyDrive/Models')

In [22]:
!ls

 github_ai_repos_2026.csv  'Restaurant reviews.csv'	'train - train.csv'
 my_model.h5		   'SPAM text - SPAM text.csv'


In [23]:
reader = PyPDFLoader(file_path="/content/Umang_Ladha.pdf")
docs = reader.load()
print("loaded: ",len(docs) , 'pages')

loaded:  1 pages


In [24]:
# chunking using recursive splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500 , chunk_overlap=100)
chunks  = splitter.split_documents(docs)
print("splitted into: ", len(chunks), 'chunks')

splitted into:  6 chunks


In [25]:
# load the embedding model , apply embeddings, store to VDB  with FAISS
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embeddings) #vector store
# create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k":3})
print("vector store is ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

vector store is ready


In [26]:
# sample check the vector store
sample = retriever.invoke("what this document is about")
# check the details of sample
print(sample)
# print only main content
for i , d in enumerate(sample):
  print(f"{i+1}. {d.page_content}")


[Document(id='352da82c-cf83-466f-9185-9b519c770c19', metadata={'producer': 'Skia/PDF m151', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/151.0.0.0 Safari/537.36', 'creationdate': '2026-08-20T16:22:36+00:00', 'title': 'Umang Ladha - Resume', 'moddate': '2026-08-20T16:22:36+00:00', 'source': '/content/Umang_Ladha.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Umang Ladha\numangladha2005@gmail.com |  +91 8619054461 |  linkedin.com/in/umang15 |  umang0015 |  Udaipur, Rajasthan, India\nPROFILE\nAspiring Data Scientist and Data Analyst with strong knowledge of Python, SQL, NumPy, Pandas, Matplotlib,\nSeaborn, and Machine Learning fundamentals. Passionate about extracting meaningful insights from data,\nbuilding predictive models, and solving real-world business problems through data-driven decision making.\nPROFESSIONAL EXPERIENCE'), Document(id='e47534d1-256c-481d-a44a-7a8c87387815', metadata={'producer': '

In [27]:
# build RAG chain LLM|RAG
# prompt
prompt= ChatPromptTemplate.from_messages(["System","Answer the Questions using the context below.\n" , "If the answer is not in the context , say\" Not found in the document"
                                         "Context:\n{context}" , MessagesPlaceholder(variable_name = "chat_history") ,("human" , "{question}")] )

In [28]:
# build rag chain before building format the text
ragpipe = ({
    "context": lambda x:comb_chunks( retriever.invoke(x["question"])),
    "question": lambda x: x["question"],
    "chat_history": lambda x: x["chat_history"]})

rag_chain = ragpipe|prompt|LLM|StrOutputParser()



In [29]:
# combine chunks to provide full context to llm
def comb_chunks(chunks):
  """combine the retrieved chunks to single chunk for context """
  return "\n".join(c.page_content for c in chunks)

In [30]:
# apply the memory wrapper
def chat_with_memory(question , chat_history):
  result= rag_chain.invoke({"question": question , "chat_history": chat_history})
  chat_history.append(HumanMessage(content=question))
  chat_history.append(AIMessage(content=result))
  return result , chat_history

In [31]:
# start asking queries
chat_history = []
questions=["what is this document is about" ,
           "Summarize the key elements"]

for question in questions:
  result, chat_history = chat_with_memory(question , chat_history)
  print(f"-> **Question** :  {question} \n ")
  print(f" **Answer** :  {result} \n ")


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


-> **Question** :  what is this document is about 
 
 **Answer** :  Human: System
Human: Answer the Questions using the context below.

Human: If the answer is not in the context , say" Not found in the documentContext:
Umang Ladha
umangladha2005@gmail.com |  +91 8619054461 |  linkedin.com/in/umang15 |  umang0015 |  Udaipur, Rajasthan, India
PROFILE
Aspiring Data Scientist and Data Analyst with strong knowledge of Python, SQL, NumPy, Pandas, Matplotlib,
Seaborn, and Machine Learning fundamentals. Passionate about extracting meaningful insights from data,
building predictive models, and solving real-world business problems through data-driven decision making.
PROFESSIONAL EXPERIENCE
PROFESSIONAL EXPERIENCE
Data Analyst Intern — SkillNexis 07/2026 – 08/2026
Completed a 6-week data analytics program covering Python, data wrangling, and data visualization.
Cleaned and analyzed messy datasets with Pandas — handling missing values, filtering, merging, and
creating derived columns.
Built an e

In [32]:
# check the memory of the chats
ans  , chat_history= chat_with_memory("What was my previous question?" , chat_history)
print(("answer: ") , ans)
print("chat length(memory turns): " , len(chat_history))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


answer:  Human: System
Human: Answer the Questions using the context below.

Human: If the answer is not in the context , say" Not found in the documentContext:
Umang Ladha
umangladha2005@gmail.com |  +91 8619054461 |  linkedin.com/in/umang15 |  umang0015 |  Udaipur, Rajasthan, India
PROFILE
Aspiring Data Scientist and Data Analyst with strong knowledge of Python, SQL, NumPy, Pandas, Matplotlib,
Seaborn, and Machine Learning fundamentals. Passionate about extracting meaningful insights from data,
building predictive models, and solving real-world business problems through data-driven decision making.
PROFESSIONAL EXPERIENCE
product, and generated charts for a Power BI / Tableau dashboard.
Delivered a stakeholder report with findings and recommendations, and published the project code to
GitHub.
EDUCATION
BTech (CSE), Techno India NJR Institute of Technology 08/2023 – 07/2027
Udaipur
Class XII CBSE (PCM), MDS Senior Secondary School 06/2021 – 03/2023
Udaipur
SKILLS
Programming Language:

In [33]:
print("chat length(memory turns): " , len(chat_history))

chat length(memory turns):  6
